Source volume: /Volumes/revenue_operations/bronze/source_files

In [0]:
sourch_path = "/Volumes/revenue_operations/bronze/source_files"

In [0]:
from pathlib import Path
dir_path = Path(sourch_path)
all_files = [str(f) for f in dir_path.rglob('*') if f.is_file()]

print(*all_files, sep = "\n")

In [0]:
import pandas as pd

customers = pd.read_csv(all_files[0])
customers.head(5)

In [0]:
customers = spark.read.csv(all_files[0], header = True, inferSchema = True)
customers.show(5)

In [0]:
customers.printSchema()

print("Number of rows in customers dataset = ", customers.count())

customers.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.bronze.customers")

In [0]:
geolocation = spark.read.csv(all_files[1], header = True, inferSchema = True)
geolocation.show(5)
geolocation.printSchema()

print("Number of rows in geolocation dataset = ", geolocation.count())

geolocation.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.bronze.geolocation")

In [0]:
order_items = spark.read.csv(all_files[2], header = True, inferSchema = True)

order_items.show(5)
order_items.printSchema()
print("Total Number of rows in order items dataset = ", order_items.count())

order_items.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.bronze.order_items")

In [0]:
payments = spark.read.csv(all_files[3], header = True, inferSchema = True)

payments.show(5)
payments.printSchema()

print("Total Number of rows in payments dataset = ", payments.count())

payments.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.bronze.payments")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Define schema explicitly to handle mixed content in review_score column
reviews_schema = StructType([
    StructField("review_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("review_score", StringType(), True),  # Read as string first
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", TimestampType(), True),
    StructField("review_answer_timestamp", TimestampType(), True)
])

reviews = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .schema(reviews_schema)
    .csv(all_files[4])
)

# Cast review_score to integer, filtering out invalid values
from pyspark.sql.functions import col
reviews = reviews.withColumn("review_score", col("review_score").cast(IntegerType()))

reviews.show(5)
reviews.printSchema()

print("Total Number off rows in reviews dataset = ", reviews.count())

reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("revenue_operations.bronze.reviews")

In [0]:
orders = spark.read.csv(all_files[5], header = True, inferSchema = True)

orders.show(5)
orders.printSchema()

print("Total Number of rows in orders dataset = ", orders.count())

orders.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.bronze.orders")

In [0]:
products = spark.read.csv(all_files[6], header = True, inferSchema = True)

products.show(5)
products.printSchema()

print("Total number of rows in prroducts dataset = ", products.count())

products.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.bronze.products")


In [0]:
sellers = spark.read.csv(all_files[7], header = True, inferSchema = True)

sellers.show(5)
sellers.printSchema()

print("Total Number of rpws in sellers dataset = ", sellers.count())

sellers.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.bronze.sellers")

In [0]:
product_category_translation = spark.read.csv(all_files[8], header = True, inferSchema = True)

product_category_translation.show(5)
product_category_translation.printSchema()

print("Total Number of rows in product category translation dataset = ", product_category_translation.count())

product_category_translation.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.bronze.product_category_tr")